In [1]:
# # setup.py
# from setuptools import setup, find_packages

# setup(
#     name="torch_spconv",
#     version="0.1.0",
#     packages=find_packages(),
#     install_requires=[
#         "torch",
#         "torch_sparse",
#         "torch_scatter",
#     ],
# )

# # torch_spconv/
# # ├── __init__.py
# # ├── functional/
# # │   ├── __init__.py
# # │   ├── conv.py          # Convolution operations
# # │   ├── pool.py          # Pooling operations
# # │   └── utils.py         # Utility functions
# # ├── nn/
# # │   ├── __init__.py
# # │   ├── conv.py          # Convolution modules
# # │   ├── pool.py          # Pooling modules
# # │   └── functional.py    # Additional functional modules
# # ├── csrc/               # C++/CUDA implementations
# # │   ├── cpu/
# # │   └── cuda/
# # └── utils/

# #     ├── __init__.py
# #     └── sparse_utils.py  # Sparse tensor utilities


In [2]:
import torch
import torch.nn as nn
import torch_scatter
from torch_sparse import SparseTensor
from typing import Tuple
import math

def sparse_conv3d(
    features: torch.Tensor,
    indices: torch.Tensor,
    weight: torch.Tensor,
    kernel_size: int = 3,
    stride: int = 1,
    padding: int = 1,
    dilation: int = 1,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Sparse 3D convolution implementation using torch_scatter
    """
    N = features.size(0)
    C_in = features.size(1)
    C_out = weight.size(0)
    K = kernel_size
    device = features.device
    
    # Create neighbor offsets for kernel
    offsets = []
    offset_range = range(-(K//2), K//2 + 1)
    for z in offset_range:
        for y in offset_range:
            for x in offset_range:
                offsets.append(torch.tensor([0, x, y, z], device=device))
    offsets = torch.stack(offsets, dim=0)  # (K^3, 4)
    
    # Compute neighborhood indices
    indices_expanded = indices.unsqueeze(1) + offsets.unsqueeze(0) * dilation  # (N, K^3, 4)
    indices_flat = indices_expanded.view(-1, 4)  # (N*K^3, 4)
    
    # Create unique index for each point based on its coordinates
    hash_multipliers = torch.tensor([1, 10000, 10000, 10000], device=device)
    indices_hash = (indices_flat * hash_multipliers).sum(dim=-1)  # (N*K^3,)
    
    # Get unique indices and mapping
    unique_hash, inverse_indices = torch.unique(indices_hash, return_inverse=True)
    
    # Create point to neighbor mapping
    point_indices = torch.arange(N, device=device).repeat_interleave(K**3)  # (N*K^3,)
    kernel_indices = torch.arange(K**3, device=device).repeat(N)  # (N*K^3,)
    
    # Gather features from neighbors
    neighbor_features = features[inverse_indices]  # (N*K^3, C_in)
    
    # Reshape to separate points and neighbors
    neighbor_features = neighbor_features.view(N, K**3, C_in)  # (N, K^3, C_in)
    
    # Reshape weight for efficient computation
    weight = weight.view(C_out, -1)  # (C_out, K^3*C_in)
    
    # Compute convolution
    x = neighbor_features.view(N, -1)  # (N, K^3*C_in)
    output_features = torch.matmul(x, weight.t())  # (N, C_out)
    
    # Update output indices for stride
    output_indices = indices.clone()
    if stride > 1:
        output_indices[:, 1:] = output_indices[:, 1:] // stride
    
    return output_features, output_indices

class SparseConvolution(nn.Module):
    """Sparse 3D Convolution module"""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = True
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        
        # Initialize weights
        std = 1.0 / math.sqrt(in_channels * kernel_size**3)
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size, kernel_size) * std
        )
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter('bias', None)

    def forward(self, x: torch.Tensor, indices: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        out_features, out_indices = sparse_conv3d(
            x, indices, self.weight,
            self.kernel_size, self.stride,
            self.padding, self.dilation
        )
        
        if self.bias is not None:
            out_features = out_features + self.bias.view(1, -1)
        
        return out_features, out_indices

In [3]:
N = 1000
in_channels = 16
out_channels = 32
kernel_size = 3

# Create random features and indices
features = torch.randn(N, in_channels)
indices = torch.cat([
    torch.zeros(N, 1),  # batch index
    torch.randint(0, 64, (N, 3))  # 3D coordinates
], dim=1).long()

# Create convolution layer
conv = SparseConvolution(
    in_channels=in_channels,
    out_channels=out_channels,
    kernel_size=kernel_size
)

# Forward pass
out_features, out_indices = conv(features, indices)

# Print shapes
print(f"Input features shape: {features.shape}")
print(f"Input indices shape: {indices.shape}")
print(f"Output features shape: {out_features.shape}")
print(f"Output indices shape: {out_indices.shape}")

# Verify output shapes
expected_output_shape = (N, out_channels)
assert out_features.shape == expected_output_shape, \
    f"Expected output shape {expected_output_shape}, got {out_features.shape}"


Input features shape: torch.Size([1000, 16])
Input indices shape: torch.Size([1000, 4])
Output features shape: torch.Size([1000, 32])
Output indices shape: torch.Size([1000, 4])


In [6]:
import torch
import torch.nn as nn
import torch_scatter
from torch_sparse import SparseTensor
from typing import Tuple
import math

def sparse_conv3d(
    features: torch.Tensor,
    indices: torch.Tensor,
    weight: torch.Tensor,
    kernel_size: int = 3,
    stride: int = 1,
    padding: int = 1,
    dilation: int = 1,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Sparse 3D convolution implementation using torch_scatter
    
    Args:
        features: (N, C_in) Input features
        indices: (N, 4) Batch index + 3D coordinates
        weight: (C_out, C_in, K, K, K) Convolution weights
        kernel_size: Size of the convolution kernel
        stride: Convolution stride
        padding: Convolution padding
        dilation: Convolution dilation
    """
    N = features.size(0)
    C_in = features.size(1)
    C_out = weight.size(0)
    K = kernel_size
    device = features.device
    
    # Apply padding to indices if needed
    if padding > 0:
        indices_padded = indices.clone()
        indices_padded[:, 1:] += padding  # Add padding to x, y, z coordinates
    else:
        indices_padded = indices
    
    # Create neighbor offsets for kernel
    offsets = []
    offset_range = range(-(K//2), K//2 + 1)
    for z in offset_range:
        for y in offset_range:
            for x in offset_range:
                offsets.append(torch.tensor([0, x, y, z], device=device))
    offsets = torch.stack(offsets, dim=0)  # (K^3, 4)
    
    # Compute neighborhood indices
    indices_expanded = indices_padded.unsqueeze(1) + offsets.unsqueeze(0) * dilation  # (N, K^3, 4)
    indices_flat = indices_expanded.view(-1, 4)  # (N*K^3, 4)
    
    # Create point to neighbor mapping
    point_indices = torch.arange(N, device=device).repeat_interleave(K**3)  # (N*K^3,)
    kernel_indices = torch.arange(K**3, device=device).repeat(N)  # (N*K^3,)
    
    # Create hash for indices to identify unique points
    hash_multipliers = torch.tensor([1, 10000, 10000, 10000], device=device)
    indices_hash = (indices_flat * hash_multipliers).sum(dim=-1)  # (N*K^3,)
    
    # Filter valid indices (within padding)
    if padding > 0:
        valid_mask = (indices_flat[:, 1:] >= 0) & (indices_flat[:, 1:] < indices_padded[:, 1:].max() + 1)
        valid_mask = valid_mask.all(dim=-1)
        indices_flat = indices_flat[valid_mask]
        indices_hash = indices_hash[valid_mask]
        point_indices = point_indices[valid_mask]
        kernel_indices = kernel_indices[valid_mask]
    
    # Get unique points and mapping
    unique_hash, inverse_indices = torch.unique(indices_hash, return_inverse=True)
    
    # Gather and aggregate features
    neighbor_features = features.new_zeros(len(unique_hash), C_in)
    neighbor_features.index_add_(0, inverse_indices, features[point_indices])
    neighbor_count = torch_scatter.scatter_add(
        torch.ones_like(inverse_indices, dtype=torch.float),
        inverse_indices,
        dim=0,
        dim_size=len(unique_hash)
    )
    neighbor_features = neighbor_features / neighbor_count.clamp(min=1).unsqueeze(-1)
    
    # Reshape features according to kernel positions
    kernel_features = torch.zeros(N, K**3, C_in, device=device)
    kernel_features[point_indices, kernel_indices] = neighbor_features[inverse_indices]
    
    # Apply convolution weights
    kernel_features = kernel_features.view(N, -1)  # (N, K^3*C_in)
    weight = weight.view(C_out, -1)  # (C_out, K^3*C_in)
    output_features = torch.matmul(kernel_features, weight.t())  # (N, C_out)
    
    # Update output indices for stride
    output_indices = indices.clone()
    if stride > 1:
        output_indices[:, 1:] = output_indices[:, 1:] // stride
    
    return output_features, output_indices

class SparseConvolution(nn.Module):
    """Sparse 3D Convolution module"""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = True
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        
        # Initialize weights
        std = 1.0 / math.sqrt(in_channels * kernel_size**3)
        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size, kernel_size) * std
        )
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter('bias', None)

    def forward(self, x: torch.Tensor, indices: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        out_features, out_indices = sparse_conv3d(
            x, indices, self.weight,
            self.kernel_size, self.stride,
            self.padding, self.dilation
        )
        
        if self.bias is not None:
            out_features = out_features + self.bias.view(1, -1)
        
        return out_features, out_indices

def test_sparse_convolution():
    """Test function to verify sparse convolution implementation"""
    torch.manual_seed(42)
    
    # Create random input
    N = 1000
    in_channels = 16
    out_channels = 32
    kernel_size = 3
    
    # Create random features and indices
    features = torch.randn(N, in_channels)
    indices = torch.cat([
        torch.zeros(N, 1),  # batch index
        torch.randint(0, 64, (N, 3))  # 3D coordinates
    ], dim=1).long()
    
    # Test different configurations
    configs = [
        {'padding': 0, 'stride': 1, 'dilation': 1},
        {'padding': 1, 'stride': 1, 'dilation': 1},
        {'padding': 1, 'stride': 20, 'dilation': 1},
        {'padding': 1, 'stride': 1, 'dilation': 2},
    ]
    
    for config in configs:
        print(f"\nTesting configuration: {config}")
        
        # Create convolution layer
        conv = SparseConvolution(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            **config
        )
        
        # Forward pass
        out_features, out_indices = conv(features, indices)
        
        # Print shapes
        print(f"Input features shape: {features.shape}")
        print(f"Input indices shape: {indices.shape}")
        print(f"Output features shape: {out_features.shape}")
        print(f"Output indices shape: {out_indices.shape}")
        
        # Basic validation
        if config['stride'] > 1:
            assert torch.all(out_indices[:, 1:] == indices[:, 1:] // config['stride'])
        
        # Check output ranges
        assert not torch.isnan(out_features).any(), "Output contains NaN values"
        assert not torch.isinf(out_features).any(), "Output contains Inf values"

In [7]:
test_sparse_convolution()


Testing configuration: {'padding': 0, 'stride': 1, 'dilation': 1}
Input features shape: torch.Size([1000, 16])
Input indices shape: torch.Size([1000, 4])
Output features shape: torch.Size([1000, 32])
Output indices shape: torch.Size([1000, 4])

Testing configuration: {'padding': 1, 'stride': 1, 'dilation': 1}
Input features shape: torch.Size([1000, 16])
Input indices shape: torch.Size([1000, 4])
Output features shape: torch.Size([1000, 32])
Output indices shape: torch.Size([1000, 4])

Testing configuration: {'padding': 1, 'stride': 20, 'dilation': 1}
Input features shape: torch.Size([1000, 16])
Input indices shape: torch.Size([1000, 4])
Output features shape: torch.Size([1000, 32])
Output indices shape: torch.Size([1000, 4])

Testing configuration: {'padding': 1, 'stride': 1, 'dilation': 2}
Input features shape: torch.Size([1000, 16])
Input indices shape: torch.Size([1000, 4])
Output features shape: torch.Size([1000, 32])
Output indices shape: torch.Size([1000, 4])
